# 8. MCH hetero

Part of the **[Fig. 3 chapter](fig3.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

In [ ]:
import numpy as np
import pandas as pd
from glob import glob
from scipy.sparse import csr_matrix
from scipy.stats import zscore
from concurrent.futures import ProcessPoolExecutor, as_completed

import os
import cooler
import pysam
import anndata
import scanpy as sc
from ALLCools.plot import *
from ALLCools.clustering import *
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [ ]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [ ]:
indir = f'{ENTEX_ROOT}/'
outdir = f'{indir}analysis/mCH_distribution/'


In [ ]:
# chrom, start, end = 'chr1', 1800000, 3800000
# chrom, start, end = 'chr2', 0, 240000000


In [ ]:
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
# L1_meta = L1_meta.drop('c7', axis=0)
L1_annot = L1_meta['L1_abbr'].to_dict()
L1_color = L1_meta['color'].to_dict()
# L1_annot['c35'] = 'Hema Bnaive'
# L1_annot['c7'] = 'Hema Bmem'
# L1_color['c35'] = L1_color['c7']


In [ ]:
nbins = 1000
reslist = pd.Index([10, 50, 100, 500, 1000, 5000, 10000, 50000])
context_list = pd.Index([f'C{xx}{yy}' for xx in 'ACGT' for yy in 'ACGT'])
selch = np.array([xx[1]!='G' for xx in context_list])

In [ ]:
# result = {ct:np.load(f'mCH_distribution/multires/{ct}_hist.npy') for ct in L1_list}
# data = pd.DataFrame([result[ct][0][selch].sum(axis=0) for ct in L1_list], index=L1_list)
# data = (data.T / data.sum(axis=1)) * 1e6


In [ ]:
# nbins = 100
# reslist = [1, 5, 10, 50, 100, 500, 1000, 5000, 10000, 50000]
palette = sns.color_palette('rainbow_r', len(reslist))


In [ ]:
def num2str(num):
    if num>=1e6:
        num_str = f'{int(num//1e6)}M'
    elif num>=1e3:
        num_str = f'{int(num//1e3)}k'
    else:
        num_str = f'{num}'
    return num_str


In [ ]:
from matplotlib.lines import Line2D

fig, ax = plt.subplots(dpi=300)
legend_lines = [Line2D([0], [0], color=palette[k], lw=2, label=num2str(reslist[k])) for k in range(len(reslist))]
ax.legend(handles=legend_lines, loc='upper right')
ax.axis('off')
# fig.savefig('mCG_distribution/rainbow_line_palette.pdf', transparent=True)


In [ ]:
palette = sns.color_palette('rainbow_r', len(reslist))
xticks = np.array([0, 0.5, 1.0])
fig, axes = plt.subplots(1, 2, figsize=(3,1), dpi=300)
for i,ct in enumerate(['c2','c5']):
    result = np.load(f'mCH_distribution/multires/{ct}_hist.npy')
    tmp = result[0, selch, :].sum(axis=0)
    nbins = (np.cumsum(tmp) < (0.95*tmp.sum())).sum()
    ax = axes[i]
    ax.set_title(L1_annot[ct], fontsize=8)
    for k in range(8):
        # ax.bar(np.arange(1000), result[f'{ct}-{bed}'][k]/result[f'{ct}-{bed}'][k].sum()*100, 
        #        width=1, color=palette[k], alpha=0.2)
        ax.plot(np.arange(nbins), result[k, selch, :nbins].sum(axis=0)/result[k, selch, :nbins].sum()*100,
                linewidth=0.5, color=palette[k], alpha=1)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
    ax.set_xticks(xticks*nbins)
    ax.set_xticklabels(np.around(xticks / (1000/nbins), decimals=3), fontsize=8)
    
fig.tight_layout()
# fig.savefig('mCG_distribution/L1_siteCG_multires_pmd_nonpmd_hist.pdf', transparent=True)


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import pysam
import cooler
import anndata
import scanpy as sc
from concurrent.futures import ProcessPoolExecutor, as_completed

posres = 1000
posres_str = '1k'

def histcg(allc_path, chunk, binmax):
    with pysam.TabixFile(allc_path) as allc:
        hist = {}
        for i,(chrom,start,end) in enumerate(zip(chunk['chrom'].values, 
                                                 chunk['start'].values, 
                                                 chunk['end'].values)):
            npos = (end - start) // posres * posres
            if npos==0:
                continue
            region = f'{chrom}-{start}-{end}'
            data = []
            allc_lines = allc.fetch(chrom, start+1, end)
            for line in allc_lines:
                _, pos, _, context, mc, cov, *_ = line.split("\t")
                if context[1] in 'ACT':
                    data.append([pos, mc, cov])

            data = pd.DataFrame(data, columns=['pos', 'mc', 'cov']).astype(int)
            
            posfilter = np.ones(data.shape[0]).astype(bool)
            for bed in rm_list:
                bedtmp = bed[chrom][np.logical_and(bed[chrom][:,0]<end, bed[chrom][:,1]>start)]
                for xx,yy in bedtmp:
                    posfilter[np.logical_and(data['pos']>=xx, data['pos']<=yy)] = False

            data = data.loc[posfilter & ((data['pos']-start) < npos)]
            data_mc = csr_matrix((data['mc'], (np.zeros(data.shape[0]), data['pos']-start-1)), shape=[1, npos])
            data_cov = csr_matrix((data['cov'], (np.zeros(data.shape[0]), data['pos']-start-1)), shape=[1, npos])
            tmp = data_mc.reshape((-1, posres)).sum(axis=1).A1 / data_cov.reshape((-1, posres)).sum(axis=1).A1
            tmp = tmp[~np.isnan(tmp)]

            if data.shape[0]>0:
                hist[region] = np.histogram(tmp, bins=nbins, range=(0,binmax))[0]
            else:
                hist[region] = np.zeros(nbins)
        return hist
    

In [ ]:
knn = 25
cpu = 32
nbins = 100
binres = 100000
binres_str = '100k'
indir = f'{ENTEX_ROOT}/'
outdir = f'{indir}analysis/mCH_distribution/{binres_str}b/'
os.makedirs(outdir, exist_ok=True)
chrom_size_path = f'{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [ ]:
rm_list = []
for bed_path in [f'{REF_ROOT}/hg38/fasta/hg38.altseq.bed', f'{REF_ROOT}/blacklist/hg38-blacklist.v2.bed.gz']:
    bed = pd.read_csv(bed_path, sep='\t', header=None, index_col=None)
    bed = {chrom:bed.loc[bed[0]==chrom, [1,2]].values for chrom in chrom_sizes.index}
    rm_list.append(bed)
    

In [ ]:
bed = cooler.util.binnify(chrom_sizes, binres)
bed = bed.loc[bed['chrom'].isin(chrom_sizes.index)]
chunk_size = bed.index.max() // 150
bed['chunk'] = bed.index // chunk_size
print(bed.shape[0], bed['chunk'].max())


In [ ]:
ct = 'c10'
allc_path = f'{indir}merged_allc/L1/CHN/{ct}.allc.tsv.gz'
result = np.load(f'mCH_distribution/multires/{ct}_hist.npy')
tmp = result[0, selch, :].sum(axis=0)
binmax = (np.cumsum(tmp) < (0.95*tmp.sum())).sum() / 1000
print(binmax)


In [ ]:
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for i,chunk in bed.groupby('chunk'):
        future = executor.submit(
            histcg,
            allc_path=allc_path, 
            chunk=chunk,
            binmax=binmax,
        )
        futures[future] = i

    result = {}
    for future in as_completed(futures):
        tmp = futures[future]
        result.update(future.result())
        print(f'chunk{tmp} finished')
        

In [ ]:
result = pd.DataFrame(result)
result = result.loc[:, (result.sum(axis=0)>0)]
result = result / result.sum(axis=0).values


In [ ]:
adata = anndata.AnnData(X=result.T)
sc.pp.neighbors(adata, n_neighbors=knn, use_rep='X')
adata.write_h5ad(f'{outdir}{ct}_{binres_str}b_{posres_str}b_hist.h5ad')


In [ ]:
ct = 'c2'
adata = anndata.read_h5ad(f'{outdir}{ct}_{binres_str}b_{posres_str}b_hist.h5ad')


In [ ]:
res = 0.5
sc.tl.leiden(adata, resolution=res, random_state=0, flavor='igraph')
adata.obs[f'leiden_{res}'] = adata.obs['leiden'].copy()


In [ ]:
nc = 2
cluster_key = f'kmeans{nc}'
# adata = anndata.read_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')
adata.obs[cluster_key] = KMeans(n_clusters=nc, random_state=0, n_init=10).fit(adata.X).labels_


In [ ]:
nc = 3
cluster_key = f'kmeans{nc}'
# res = 0.5
# cluster_key = f'leiden_{res}'
leg = np.sort(adata.obs[cluster_key].unique())
nc = len(leg)
cluster_color = sns.color_palette('Set2', nc)

fig, axes = plt.subplots(1, 2, figsize=(2,1), sharey='all', dpi=300, 
                         gridspec_kw={'width_ratios': [20,1]})

ax = axes[0]
sns.despine(ax=ax, left=True, bottom=True)
# adata = anndata.read_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')
# selc = np.random.choice(np.arange(adata.shape[0]), 1000, False)
# idx = adata.obs[cluster_key].iloc[selc].sort_values().index
idx = adata.obs[cluster_key].sort_values().index
tmp = adata[idx].X
ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', 
          interpolation='none', rasterized=True)

offset = list(adata.obs.loc[idx, cluster_key].value_counts().loc[leg].cumsum())
offset = [0] + offset
# ax.set_xticks([-0.5, 49.5, 99.5])
# ax.set_xticklabels([0, 0.5, 1])
ax.set_yticks([])

ax = axes.flatten()[1]
ax.axis('off')
for k in range(nc):
    rect = patches.Rectangle((0, offset[k]), 1, offset[k+1]-offset[k], linewidth=0, edgecolor='none', facecolor=cluster_color[k])
    ax.add_patch(rect)
    # ax.text(np.mean(offset[i:(i+2)]), -0.2, label, rotation=90, fontsize=10, horizontalalignment='left', verticalalignment='top')
    
fig.tight_layout()


In [ ]:
binres_str = '100k'
posres_str = '1k'
res = 0.5
cluster_key = f'leiden_{res}'

fig, axes = plt.subplots(6, 12, figsize=(13,12), sharey='all', sharex='col', dpi=300, 
                         gridspec_kw={'width_ratios': np.tile([20,1], 6)})

for i,ct in enumerate(L1_meta.drop(['c35','c36'], axis=0).index):
    ax = axes.flatten()[i*2]
    sns.despine(ax=ax, left=True, bottom=True)
    adata = anndata.read_h5ad(f'{outdir}{binres_str}b/{ct}_{binres_str}b_{posres_str}b_hist.h5ad')
    adata.obs['mean'] = np.mean((adata.X * np.arange(100)), axis=1)
    # adata.obs['group_mean'] = adata.obs.groupby(cluster_key)['mean'].mean().loc[adata.obs[cluster_key]].values
    # idx = adata.obs.sort_values(['group_mean', 'mean']).index
    idx = adata.obs.sort_values('mean').index
    tmp = adata[idx].X
    ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', 
              interpolation='none', rasterized=True)
    ax.set_title(L1_annot[ct])

    leg = np.sort(adata.obs[cluster_key].unique())
    nc = len(leg)
    cluster_color = sns.color_palette('Set2', nc)
    offset = list(adata.obs.loc[idx, cluster_key].value_counts().loc[leg].cumsum())
    offset = [0] + offset
    # ax.set_xticks([-0.5, 49.5, 99.5])
    # ax.set_xticklabels([0, 0.5, 1])
    ax.set_yticks([])

    ax = axes.flatten()[i*2+1]
    ax.axis('off')
    for k in range(nc):
        rect = patches.Rectangle((0, offset[k]), 1, offset[k+1]-offset[k], linewidth=0, edgecolor='none', facecolor=cluster_color[k])
        ax.add_patch(rect)
        # ax.text(np.mean(offset[i:(i+2)]), -0.2, label, rotation=90, fontsize=10, horizontalalignment='left', verticalalignment='top')

fig.tight_layout()


In [ ]:
binres_str = '100k'
posres_str = '100'
res = 0.5
cluster_key = f'leiden_{res}'

fig, axes = plt.subplots(6, 12, figsize=(13,12), sharey='all', sharex='col', dpi=300, 
                         gridspec_kw={'width_ratios': np.tile([20,1], 6)})

for i,ct in enumerate(L1_meta.drop(['c35','c36'], axis=0).index):
    ax = axes.flatten()[i*2]
    sns.despine(ax=ax, left=True, bottom=True)
    adata = anndata.read_h5ad(f'{outdir}{binres_str}b/{ct}_{binres_str}b_{posres_str}b_hist.h5ad')
    adata.obs['mean'] = np.mean((adata.X * np.arange(100)), axis=1)
    # adata.obs['group_mean'] = adata.obs.groupby(cluster_key)['mean'].mean().loc[adata.obs[cluster_key]].values
    # idx = adata.obs.sort_values(['group_mean', 'mean']).index
    idx = adata.obs.sort_values('mean').index
    tmp = adata[idx].X
    ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', 
              interpolation='none', rasterized=True)
    ax.set_title(L1_annot[ct])

    leg = np.sort(adata.obs[cluster_key].unique())
    nc = len(leg)
    cluster_color = sns.color_palette('Set2', nc)
    offset = list(adata.obs.loc[idx, cluster_key].value_counts().loc[leg].cumsum())
    offset = [0] + offset
    # ax.set_xticks([-0.5, 49.5, 99.5])
    # ax.set_xticklabels([0, 0.5, 1])
    ax.set_yticks([])

    ax = axes.flatten()[i*2+1]
    ax.axis('off')
    for k in range(nc):
        rect = patches.Rectangle((0, offset[k]), 1, offset[k+1]-offset[k], linewidth=0, edgecolor='none', facecolor=cluster_color[k])
        ax.add_patch(rect)
        # ax.text(np.mean(offset[i:(i+2)]), -0.2, label, rotation=90, fontsize=10, horizontalalignment='left', verticalalignment='top')

fig.tight_layout()


In [ ]:
## c1
# nc = 2
# cluster_key = f'kmeans{nc}'
binres_str = '10k'
posres_str = '100'
res = 0.5
cluster_key = f'leiden_{res}'

fig, axes = plt.subplots(6, 12, figsize=(13,12), sharey='all', sharex='col', dpi=300, 
                         gridspec_kw={'width_ratios': np.tile([20,1], 6)})

for i,ct in enumerate(L1_meta.drop(['c35','c36'], axis=0).index):
    ax = axes.flatten()[i*2]
    sns.despine(ax=ax, left=True, bottom=True)
    adata = anndata.read_h5ad(f'{outdir}{binres_str}b/{ct}_{binres_str}b_{posres_str}b_hist.h5ad')
    adata.obs['mean'] = np.mean((adata.X * np.arange(100)), axis=1)
    # adata.obs['group_mean'] = adata.obs.groupby(cluster_key)['mean'].mean().loc[adata.obs[cluster_key]].values
    # idx = adata.obs.sort_values(['group_mean', 'mean']).index
    idx = adata.obs.sort_values('mean').index
    tmp = adata[idx].X
    ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', 
              interpolation='none', rasterized=True)
    ax.set_title(L1_annot[ct])

    leg = np.sort(adata.obs[cluster_key].unique())
    nc = len(leg)
    cluster_color = sns.color_palette('Set2', nc)
    offset = list(adata.obs.loc[idx, cluster_key].value_counts().loc[leg].cumsum())
    offset = [0] + offset
    # ax.set_xticks([-0.5, 49.5, 99.5])
    # ax.set_xticklabels([0, 0.5, 1])
    ax.set_yticks([])

    ax = axes.flatten()[i*2+1]
    ax.axis('off')
    for k in range(nc):
        rect = patches.Rectangle((0, offset[k]), 1, offset[k+1]-offset[k], linewidth=0, edgecolor='none', facecolor=cluster_color[k])
        ax.add_patch(rect)
        # ax.text(np.mean(offset[i:(i+2)]), -0.2, label, rotation=90, fontsize=10, horizontalalignment='left', verticalalignment='top')

fig.tight_layout()


In [ ]:
## c1
# nc = 2
# cluster_key = f'kmeans{nc}'
binres_str = '100k'
posres_str = '10'
res = 0.5
cluster_key = f'leiden_{res}'

fig, axes = plt.subplots(6, 12, figsize=(13,12), sharey='all', sharex='col', dpi=300, 
                         gridspec_kw={'width_ratios': np.tile([20,1], 6)})

for i,ct in enumerate(L1_meta.drop(['c35','c36'], axis=0).index):
    ax = axes.flatten()[i*2]
    sns.despine(ax=ax, left=True, bottom=True)
    adata = anndata.read_h5ad(f'{outdir}{binres_str}b/{ct}_{binres_str}b_{posres_str}b_hist.h5ad')
    adata.obs['mean'] = np.mean((adata.X * np.arange(100)), axis=1)
    # adata.obs['group_mean'] = adata.obs.groupby(cluster_key)['mean'].mean().loc[adata.obs[cluster_key]].values
    # idx = adata.obs.sort_values(['group_mean', 'mean']).index
    idx = adata.obs.sort_values('mean').index
    tmp = adata[idx].X
    ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', 
              interpolation='none', rasterized=True)
    ax.set_title(L1_annot[ct])

    leg = np.sort(adata.obs[cluster_key].unique())
    nc = len(leg)
    cluster_color = sns.color_palette('Set2', nc)
    offset = list(adata.obs.loc[idx, cluster_key].value_counts().loc[leg].cumsum())
    offset = [0] + offset
    # ax.set_xticks([-0.5, 49.5, 99.5])
    # ax.set_xticklabels([0, 0.5, 1])
    ax.set_yticks([])

    ax = axes.flatten()[i*2+1]
    ax.axis('off')
    for k in range(nc):
        rect = patches.Rectangle((0, offset[k]), 1, offset[k+1]-offset[k], linewidth=0, edgecolor='none', facecolor=cluster_color[k])
        ax.add_patch(rect)
        # ax.text(np.mean(offset[i:(i+2)]), -0.2, label, rotation=90, fontsize=10, horizontalalignment='left', verticalalignment='top')

fig.tight_layout()


In [ ]:
## c1
# nc = 2
# cluster_key = f'kmeans{nc}'
binres_str = '10k'
posres_str = '10'
res = 0.5
cluster_key = f'leiden_{res}'

fig, axes = plt.subplots(6, 12, figsize=(13,12), sharey='all', sharex='col', dpi=300, 
                         gridspec_kw={'width_ratios': np.tile([20,1], 6)})

for i,ct in enumerate(L1_meta.drop(['c35','c36'], axis=0).index):
    ax = axes.flatten()[i*2]
    sns.despine(ax=ax, left=True, bottom=True)
    adata = anndata.read_h5ad(f'{outdir}{binres_str}b/{ct}_{binres_str}b_{posres_str}b_hist.h5ad')
    adata.obs['mean'] = np.mean((adata.X * np.arange(100)), axis=1)
    # adata.obs['group_mean'] = adata.obs.groupby(cluster_key)['mean'].mean().loc[adata.obs[cluster_key]].values
    # idx = adata.obs.sort_values(['group_mean', 'mean']).index
    idx = adata.obs.sort_values('mean').index
    tmp = adata[idx].X
    ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', 
              interpolation='none', rasterized=True)
    ax.set_title(L1_annot[ct])

    leg = np.sort(adata.obs[cluster_key].unique())
    nc = len(leg)
    cluster_color = sns.color_palette('Set2', nc)
    offset = list(adata.obs.loc[idx, cluster_key].value_counts().loc[leg].cumsum())
    offset = [0] + offset
    # ax.set_xticks([-0.5, 49.5, 99.5])
    # ax.set_xticklabels([0, 0.5, 1])
    ax.set_yticks([])

    ax = axes.flatten()[i*2+1]
    ax.axis('off')
    for k in range(nc):
        rect = patches.Rectangle((0, offset[k]), 1, offset[k+1]-offset[k], linewidth=0, edgecolor='none', facecolor=cluster_color[k])
        ax.add_patch(rect)
        # ax.text(np.mean(offset[i:(i+2)]), -0.2, label, rotation=90, fontsize=10, horizontalalignment='left', verticalalignment='top')

fig.tight_layout()


In [ ]:
nc = 4
cluster_key = f'kmeans{nc}'
binres_str = '10k'
posres_str = '1'

fig, axes = plt.subplots(6, 12, figsize=(13,12), sharey='all', sharex='col', dpi=300, 
                         gridspec_kw={'width_ratios': np.tile([20,1], 6)})

for i,ct in enumerate(L1_meta.drop(['c7'], axis=0).index):
    ax = axes.flatten()[i*2]
    sns.despine(ax=ax, left=True, bottom=True)
    adata = anndata.read_h5ad(f'{ENTEX_ROOT}/analysis/PMD/10kb/{ct}_{binres_str}b_hist.h5ad')
    adata.obs['mean'] = np.mean((adata.X * np.arange(100)), axis=1)
    # adata.obs['group_mean'] = adata.obs.groupby(cluster_key)['mean'].mean().loc[adata.obs[cluster_key]].values
    # idx = adata.obs.sort_values(['group_mean', 'mean']).index
    idx = adata.obs.sort_values('mean').index
    tmp = adata[idx].X
    ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', 
              interpolation='none', rasterized=True)
    ax.set_title(L1_annot[ct])

    leg = np.sort(adata.obs[cluster_key].unique())
    nc = len(leg)
    cluster_color = sns.color_palette('Set2', nc)
    offset = list(adata.obs.loc[idx, cluster_key].value_counts().loc[leg].cumsum())
    offset = [0] + offset
    # ax.set_xticks([-0.5, 49.5, 99.5])
    # ax.set_xticklabels([0, 0.5, 1])
    ax.set_yticks([])

    ax = axes.flatten()[i*2+1]
    ax.axis('off')
    for k in range(nc):
        rect = patches.Rectangle((0, offset[k]), 1, offset[k+1]-offset[k], linewidth=0, edgecolor='none', facecolor=cluster_color[k])
        ax.add_patch(rect)
        # ax.text(np.mean(offset[i:(i+2)]), -0.2, label, rotation=90, fontsize=10, horizontalalignment='left', verticalalignment='top')

fig.tight_layout()


In [ ]:
data = []
for ct in L1_meta.index:
    if ct=='c7':
        continue
    tmp = anndata.read_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')
    tmp = pd.DataFrame(tmp.X, index=tmp.obs.index, 
                       columns=ct + '-' + tmp.var.index.astype(str))
    data.append(tmp)
    print(ct, tmp.shape[0])

data = pd.concat(data, axis=1, join='inner')


In [ ]:
adata = anndata.AnnData(X=data)
model = PCA(n_components=50, svd_solver='arpack', random_state=0)
adata.obsm['pca_all'] = model.fit_transform(adata.X)
npc = significant_pc_test(adata, obsm='pca_all', p_cutoff=0.01, update=False)


In [ ]:
npc = 20
knn = 25
adata.obsm['X_pca'] = adata.obsm['pca_all'][:, :npc].copy()
sc.pp.neighbors(adata, n_neighbors=knn, use_rep='X_pca')


In [ ]:
ct = 'merged'
tsne(adata, obsm='X_pca', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
adata.obsm[f'pc{npc}_tsne'] = adata.obsm['X_tsne'].copy()
adata.write_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')


In [ ]:
sc.tl.leiden(adata, resolution=0.1, random_state=0, flavor='igraph')


In [ ]:
ct = 'merged'


In [ ]:
ds = 0.1
npc = 20
coord_base = 'tsne'
dump_embedding(adata, coord_base)

fig, ax = plt.subplots(figsize=(5, 4), dpi=300, constrained_layout=True)

tmp = adata.obs.copy()
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='leiden',
                        text_anno='leiden',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )



In [ ]:
ds = 0.1
npc = 20
coord_base = 'tsne'
dump_embedding(adata, coord_base)

fig, ax = plt.subplots(figsize=(5, 4), dpi=300, constrained_layout=True)

tmp = adata.obs.copy()
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='kmeans4raw',
                        text_anno='kmeans4raw',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )



In [ ]:
# from scipy.cluster.hierarchy import linkage, fcluster

# nc = 4
# Z = linkage(result.T, metric='euclidean', method='ward')
# label = fcluster(Z, t=nc, criterion='maxclust')
# count = pd.Series(label).value_counts()
# selclust = count.index[count>100]


In [ ]:
ct = 'merged'
adata = anndata.read_h5ad(f'PMD/10kb/{ct}_10kb_hist.h5ad')
adata

In [ ]:
nc = 4
cluster_key = f'kmeans{nc}raw'
adata.obs[cluster_key] = KMeans(n_clusters=nc, random_state=0, n_init=10).fit(adata.X).labels_
count = adata.obs[cluster_key].value_counts()


In [ ]:
fig, ax = plt.subplots(figsize=(6,3), dpi=300)
sns.despine(ax=ax, left=True, bottom=True)
# leg = count.index[:20]
# if count.shape[0]>20:
#     adata.obs.loc[~adata.obs[cluster_key].isin(leg[:-1]), cluster_key] = leg[-1]
color_palette = {xx:yy for xx,yy in zip(np.sort(leg), sns.color_palette('tab20', len(leg)))}
selc = np.random.choice(np.arange(adata.shape[0]), 1000, False)
idx = adata.obs[cluster_key].iloc[selc].sort_values().index
tmp = adata[idx].X
offset = adata.obs.loc[idx, cluster_key].value_counts().loc[np.sort(leg)].cumsum()
ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', rasterized=True)
ax.set_xticks(np.arange(L1_meta.shape[0]) * 100 + 49.5)
ax.set_xticklabels(L1_meta.index.map(L1_annot), rotation=90)
ax.set_yticks(offset-0.5)
ax.set_yticklabels(np.sort(leg))


In [ ]:
rename_cluster = {0:1, 1:3, 2:2, 3:0}
adata.obs[cluster_key] = adata.obs[cluster_key].map(rename_cluster)
fig, ax = plt.subplots(figsize=(6,3), dpi=300)
sns.despine(ax=ax, left=True, bottom=True)
selc = np.random.choice(np.arange(adata.shape[0]), 1000, False)
idx = adata.obs[cluster_key].iloc[selc].sort_values().index
tmp = adata[idx].X
offset = adata.obs.loc[idx, cluster_key].value_counts().loc[np.sort(leg)].cumsum()
ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', rasterized=True)
ax.set_xticks(np.arange(L1_meta.shape[0]) * 100 + 49.5)
ax.set_xticklabels(L1_meta.index.map(L1_annot), rotation=90)
ax.set_yticks(offset-0.5)
ax.set_yticklabels(np.sort(leg))


In [ ]:
adata.obs[cluster_key].to_hdf(f'PMD/10kb/{ct}_10kb_{cluster_key}.hdf', key='data')

In [ ]:
nc = 5
cluster_key = f'kmeans{nc}raw'
adata.obs[cluster_key] = KMeans(n_clusters=nc, random_state=0, n_init=10).fit(adata.X).labels_
count = adata.obs[cluster_key].value_counts()

fig, ax = plt.subplots(figsize=(6,3), dpi=300)
sns.despine(ax=ax, left=True, bottom=True)
leg = count.index[:20]
if count.shape[0]>20:
    adata.obs.loc[~adata.obs[cluster_key].isin(leg[:-1]), cluster_key] = leg[-1]
color_palette = {xx:yy for xx,yy in zip(np.sort(leg), sns.color_palette('tab20', len(leg)))}
selc = np.random.choice(np.arange(adata.shape[0]), 1000, False)
idx = adata.obs[cluster_key].iloc[selc].sort_values().index
tmp = adata[idx].X
offset = adata.obs.loc[idx, cluster_key].value_counts().loc[np.sort(leg)].cumsum()
ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', rasterized=True)
ax.set_xticks(np.arange(L1_meta.shape[0]) * 100 + 49.5)
ax.set_xticklabels(L1_meta.index.map(L1_annot), rotation=90)
ax.set_yticks(offset-0.5)
ax.set_yticklabels(np.sort(leg))


In [ ]:
nc = 10
cluster_key = f'kmeans{nc}raw'
adata.obs[cluster_key] = KMeans(n_clusters=nc, random_state=0, n_init=10).fit(adata.X).labels_
count = adata.obs[cluster_key].value_counts()

fig, ax = plt.subplots(figsize=(6,3), dpi=300)
sns.despine(ax=ax, left=True, bottom=True)
leg = count.index[:20]
if count.shape[0]>20:
    adata.obs.loc[~adata.obs[cluster_key].isin(leg[:-1]), cluster_key] = leg[-1]
color_palette = {xx:yy for xx,yy in zip(np.sort(leg), sns.color_palette('tab20', len(leg)))}
selc = np.random.choice(np.arange(adata.shape[0]), 1000, False)
idx = adata.obs[cluster_key].iloc[selc].sort_values().index
tmp = adata[idx].X
offset = adata.obs.loc[idx, cluster_key].value_counts().loc[np.sort(leg)].cumsum()
ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', rasterized=True)
ax.set_xticks(np.arange(L1_meta.shape[0]) * 100 + 49.5)
ax.set_xticklabels(L1_meta.index.map(L1_annot), rotation=90)
ax.set_yticks(offset-0.5)
ax.set_yticklabels(np.sort(leg))


In [ ]:
outdir

In [ ]:
## merged
selc = np.random.choice(np.arange(adata.shape[0]), 1000, False)
sns.clustermap(data.iloc[selc], cmap='vlag', z_score=1, #col_cluster=False, 
               metric='euclidean', method='ward', vmax=10, # vmax=np.percentile(adata.X, 99),
               row_colors=[color_palette[xx] for xx in adata.obs['leiden'].values[selc]])


In [ ]:
def clustering(ct):
    nc = 4
    cluster_key = f'kmeans{nc}'
    adata = anndata.read_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')
    # adata.obs[cluster_key] = KMeans(n_clusters=nc, random_state=0, n_init=10).fit(adata.X).labels_
    mcg = pd.Series(np.arange(100).dot(adata.X.T), index=adata.obs.index)
    # mcg = pd.Series(adata.X.argmax(axis=1), index=adata.obs.index)
    mcg = mcg.groupby(adata.obs[cluster_key]).mean()
    rename_cluster = mcg.sort_values().reset_index().reset_index().set_index(cluster_key)['index']
    count = adata.obs[cluster_key].value_counts().loc[rename_cluster.index]
    if count.iloc[1]>count.iloc[2]:
        rename_cluster.iloc[[1, 2]] = [2, 1]
    rename_cluster = rename_cluster.sort_values()
    adata.obs[cluster_key] = adata.obs[cluster_key].map(rename_cluster)
    adata.write_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')
    return

cpu = 20
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for i,ct in enumerate(L1_meta.index):
        future = executor.submit(
            clustering,
            ct=ct,
        )
        futures[future] = ct

    for future in as_completed(futures):
        ct = futures[future]
        tmp = future.result()
        print(f'{ct} finished')


In [ ]:
nc = 4
cluster_key = f'kmeans{nc}'
cluster_color = sns.color_palette('Set2', nc)
leg_order = meta.groupby('L1')['mCGFrac'].median().sort_values().index
fig, axes = plt.subplots(6, 12, figsize=(13,12), sharey='all', sharex='col', dpi=300, 
                         gridspec_kw={'width_ratios': np.tile([20,1], 6)})
for i,ct in enumerate(leg_order):
    ax = axes.flatten()[i*2]
    sns.despine(ax=ax, left=True, bottom=True)
    adata = anndata.read_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')
    
    leg = np.sort(adata.obs[cluster_key].unique())
    # selc = np.random.choice(np.arange(adata.shape[0]), 1000, False)
    # idx = adata.obs[cluster_key].iloc[selc].sort_values().index
    idx = adata.obs[cluster_key].sort_values().index
    tmp = adata[idx].X
    ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', 
              interpolation='none', rasterized=True)
    ax.set_title(L1_annot[ct])
    offset = list(adata.obs.loc[idx, cluster_key].value_counts().loc[leg].cumsum())
    offset = [0] + offset
    ax.set_xticks([-0.5, 49.5, 99.5])
    ax.set_xticklabels([0, 0.5, 1])
    ax.set_yticks([])
    
    ax = axes.flatten()[i*2+1]
    ax.axis('off')
    for k in range(4):
        rect = patches.Rectangle((0, offset[k]), 1, offset[k+1]-offset[k], linewidth=0, edgecolor='none', facecolor=cluster_color[k])
        ax.add_patch(rect)
        # ax.text(np.mean(offset[i:(i+2)]), -0.2, label, rotation=90, fontsize=10, horizontalalignment='left', verticalalignment='top')
    
fig.tight_layout()
fig.savefig('mCG_distribution/L1_10kb_hist_kmeans4.pdf', transparent=True)


In [ ]:
import pysam

def load_allc(ct, chrom, start, end):
    global indir
    allc_path = f'{indir}merged_allc/L1/CGN/{ct}.CGN-Merge.allc.tsv.gz'
    idx, data_mc, data_cov = [], [], []
    with pysam.TabixFile(allc_path) as allc:
        allc_lines = allc.fetch(chrom, start, end)
        for line in allc_lines:
            _, pos, _, context, mc, cov, *_ = line.split("\t")
            pos, mc, cov = int(pos), int(mc), int(cov)
            idx.append(pos-start)
            data_mc.append(mc)
            data_cov.append(cov)
    return  np.array([idx, data_mc, data_cov])

def plot_bw(ct, ymin, nbins, ax):
    res = (end-start+1) // nbins
    idx, tmp_mc, tmp_cov = load_allc(ct, chrom, start, end)
    npos = len(idx)
    mc_tmp, cov_tmp = np.zeros((2, end-start+1))
    mc_tmp[idx] = tmp_mc
    cov_tmp[idx] = tmp_cov
    mc_tmp = mc_tmp[:res*nbins]
    cov_tmp = cov_tmp[:res*nbins]
    tmp = mc_tmp.reshape((-1, res)).sum(axis=1) / cov_tmp.reshape((-1, res)).sum(axis=1)
    x = np.arange(len(tmp))
    # ax.plot(x, tmp, linewidth=0.01)
    ax.set_xlim([0, nbins])
    ax.set_ylim([ymin, 1.0])
    step = nbins//4
    ax.set_xticks(np.arange(0, nbins+1, step))
    ax.set_xticklabels([f'{xx//1e6}M' for xx in np.arange(start, end+1, step*res)], fontsize=10)
    ax.set_yticks([ymin, 1.0])
    # ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
    ax.fill_between(x, tmp, ymin, where=tmp >= ymin, facecolor=L1_color[ct], interpolate=True)
    ax.set_title(L1_annot[ct], fontsize=10)
    return res


In [ ]:
def plot_bed(bed_path, start, end, res, ax):
    bed_df = pd.read_csv(bed_path, header=None, index_col=None, sep='\t', usecols=(0,1,2), names=['chrom','start','end'])
    bed_df = bed_df.loc[(bed_df['chrom']==chrom) & (bed_df['end']>=start) & (bed_df['start']<=end), ['start', 'end']]
    bed_df = (bed_df - start) // res
    print(bed_df.shape)
    for xx,yy in bed_df.values:
        rect = patches.Rectangle((xx, 0), yy-xx, 1, linewidth=0, edgecolor='none', facecolor='k')
        ax.add_patch(rect)
    ax.set_ylim([-1, 2])
    ax.set_yticks([])
    return


In [ ]:
nbins = 2000
chrom = 'chr14'
start = 44000000
end = 64000000

fig, axes = plt.subplots(6, 1, figsize=(10,2), dpi=300, sharex='all', 
                         gridspec_kw={'height_ratios': np.tile([4,1,1], 2)})
ct = 'c35'
ax = axes[0]
res = plot_bw(ct=ct, ymin=0.5, nbins=nbins, ax=ax)
ax = axes[1]
bed_path = f'{outdir}{ct}_hist_kmeans4.bed'
plot_bed(bed_path, start, end, res, ax)
ax = axes[2]
bed_path = f'{outdir}../{L1_annot[ct].replace(" ","-")}.pmd.bed'
plot_bed(bed_path, start, end, res, ax)

ct = 'c1'
ax = axes[3]
res = plot_bw(ct=ct, ymin=0.5, nbins=nbins, ax=ax)
ax = axes[4]
bed_path = f'{outdir}{ct}_hist_kmeans4.bed'
plot_bed(bed_path, start, end, res, ax)
ax = axes[5]
bed_path = f'{outdir}../{L1_annot[ct].replace(" ","-")}.pmd.bed'
plot_bed(bed_path, start, end, res, ax)

fig.tight_layout()
fig.savefig('mCG_distribution/pmd_method_compare.pdf', transparent=True)


In [ ]:
import anndata

ct = 'c1'
adata = anndata.read_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')
adata.obs[['chrom','start','end']] = adata.obs.reset_index()['index'].str.split('-', expand=True).values
adata.obs[['start','end']] = adata.obs[['start','end']].astype(int)


In [ ]:
resm = 10000
chrom, start, end = 'chr14', 44000000, 54000000

adatatmp = adata[(adata.obs['chrom']==chrom) & (adata.obs['start']>=start) & (adata.obs['end']<=end)].copy()
mccomp = np.zeros(((end-start)//resm, 100))
mccomp[(adatatmp.obs['start'] - start)//resm] = adatatmp.X.copy()


In [ ]:
label = np.zeros((end-start)//resm) + 4
label[(adatatmp.obs['start']-start)//resm] = adatatmp.obs['kmeans4'].values
label = label.astype(int)

In [ ]:
cluster_color = sns.color_palette('Set2', 4)
cluster_color = cluster_color + [(1,1,1)]
compcolor = np.array([cluster_color[xx] for xx in label])


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7, 1), dpi=300, gridspec_kw={'height_ratios': [1,0.2]})

ax = axes[0,0]
ax.axis('off')
vmax = np.percentile(mccomp,99)
ax.imshow(mccomp.T[::-1], cmap='vlag', aspect='auto', vmax=vmax, rasterized=True)
ax.set_xlim([-0.5, (end-start)//resm-0.5])

ax = axes[1,0]
ax.axis('off')
ax.imshow([compcolor], aspect='auto', rasterized=True)
ax.set_xlim([-0.5, (end-start)//resm-0.5])

order = np.argsort(label)
ax = axes[0,1]
ax.axis('off')
vmax = np.percentile(mccomp,99)
ax.imshow(mccomp.T[::-1, order], cmap='vlag', aspect='auto', vmax=vmax, rasterized=True)
ax.set_xlim([-0.5, (end-start)//resm-0.5])

ax = axes[1,1]
ax.axis('off')
ax.imshow([compcolor[order]], aspect='auto', rasterized=True)
ax.set_xlim([-0.5, (end-start)//resm-0.5])

fig.tight_layout()
fig.savefig('mCG_distribution/pmd_demonstration.pdf', transparent=True)


In [ ]:
4.9842+(0.7327/2)

In [ ]:
leg = [0, 1, 2, 3]
selc = adata.obs.index.copy()


In [ ]:
nc = 4
cluster_key = f'kmeans{nc}'
label = {}
for ct in L1_meta.index:
    adata = anndata.read_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')[selc]
    label[ct] = adata.obs[cluster_key]


In [ ]:
data = []
for pmd_ct in L1_meta.index:
    datatmp = []
    for data_ct in L1_meta.index:
        adata = anndata.read_h5ad(f'{outdir}{data_ct}_10kb_hist.h5ad')[selc]
        tmp = pd.DataFrame(adata.X, index=adata.obs.index).groupby(label[pmd_ct]).mean().loc[leg]
        datatmp.append(tmp)
    data.append(pd.concat(datatmp, axis=1))
    print(pmd_ct)

data = pd.concat(data, axis=0)


In [ ]:
data.columns = np.repeat(L1_meta.index, 100).astype(str) + '-' + data.columns.astype(str)
data.to_hdf('mCG_distribution/L1_10kb_hist_kmeans4_crossL1.hdf', key='data')

In [ ]:
color = 'r'
row_per_ct = 4
col_per_ct = 100

fig, axes = plt.subplots(1, L1_meta.shape[0], figsize=(12,6), dpi=300, gridspec_kw={'wspace':0.1})
for i,ct in enumerate(L1_meta.index):
    ax = axes[i]
    sns.despine(ax=ax, left=True, bottom=True)
    ax.imshow(data.iloc[:, (i*col_per_ct):((i+1)*col_per_ct)], cmap='vlag', aspect='auto', vmin=0, vmax=0.2, rasterized=True, interpolation='none')
    ax.set_xticks([49.5])
    ax.set_xticklabels([L1_annot[ct]], rotation=90)
    ax.set_yticks([])
    ax.plot([-0.5, col_per_ct-0.5], [i*row_per_ct-0.5, i*row_per_ct-0.5], 'k', linewidth=0.5, zorder=10)
    ax.plot([-0.5, col_per_ct-0.5], [(i+1)*row_per_ct-0.5, (i+1)*row_per_ct-0.5], 'k', linewidth=0.5, zorder=10)
    
ax = axes[0]
ax.set_yticks(np.arange(L1_meta.shape[0])*row_per_ct+0.5*row_per_ct-0.5)
ax.set_yticklabels(L1_meta.index.map(L1_annot))
ax.set_ylabel('PMD majortype')
ax = axes[int(L1_meta.shape[0]//2)]
ax.set_xlabel('mCG majortype')

fig.tight_layout()
fig.savefig('mCG_distribution/L1_10kb_hist_kmeans4_crossL1.pdf', transparent=True)


In [ ]:
color = 'r'
row_per_ct = 4
col_per_ct = 100

fig, ax = plt.subplots(figsize=(9,6), dpi=300)
ax.imshow(data, cmap='vlag', aspect='auto', vmin=0, vmax=0.2, rasterized=True, interpolation='none')
ax.set_xticks(np.arange(L1_meta.shape[0])*col_per_ct+0.5*col_per_ct-0.5)
ax.set_xticklabels(L1_meta.index.map(L1_annot), rotation=90)
ax.set_yticks(np.arange(L1_meta.shape[0])*row_per_ct+0.5*row_per_ct-0.5)
ax.set_yticklabels(L1_meta.index.map(L1_annot))
for i,bw_ct in enumerate(L1_meta.index):
    xx = col_per_ct*i-0.5
    yy = row_per_ct*i-0.5
    ax.plot([-0.5, data.shape[1]-0.5], [yy, yy], 'k')
    ax.plot([xx, xx], [yy, yy+row_per_ct], color, linewidth=0.5, zorder=10)
    ax.plot([xx+col_per_ct, xx+col_per_ct], [yy, yy+row_per_ct], color, linewidth=0.5, zorder=10)
    ax.plot([xx, xx+col_per_ct], [yy, yy], color, linewidth=0.5, zorder=10)
    ax.plot([xx, xx+col_per_ct], [yy+row_per_ct, yy+row_per_ct], color, linewidth=0.5, zorder=10)
    # data.append(datatmp)

ax.set_xlabel('mCG majortype')
ax.set_ylabel('PMD majortype')

fig.tight_layout()
# fig.savefig(f'{outdir}L1_10kb_hist_kmeans4_crossL1.pdf', transparent=True)


In [ ]:
import matplotlib.patches as patches

nc = 4
cluster_key = f'kmeans{nc}'
cluster_color = sns.color_palette('Set2', nc)
# leg_order = meta.groupby('L1')['mCGFrac'].median().sort_values().index

fig, axes = plt.subplots(2, 3, figsize=(4.2, 4), sharey='all', sharex='col', dpi=300, 
                         gridspec_kw={'width_ratios': [20,20,1]})

label = {}
for i,pmd_ct in enumerate(['c35', 'c36']):
    adata = anndata.read_h5ad(f'{outdir}{pmd_ct}_10kb_hist.h5ad')[selc]
    # label = adata.obs[cluster_key]
    idx = adata.obs[cluster_key].sort_values().index
    offset = list(adata.obs.loc[idx, cluster_key].value_counts().loc[leg].cumsum())
    offset = [0] + offset
    
    for j,data_ct in enumerate(['c35', 'c36']):
        ax = axes[i, j]
        sns.despine(ax=ax, left=True, bottom=True)
        adata = anndata.read_h5ad(f'{outdir}{data_ct}_10kb_hist.h5ad')
        tmp = adata[idx].X
        ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', 
                  interpolation='none', rasterized=True)
        if i==0:
            ax.set_title(L1_annot[data_ct])
        ax.set_xticks([-0.5, 49.5, 99.5])
        ax.set_xticklabels([0, 0.5, 1])
        ax.set_yticks([])
    
    ax = axes[i, -1]
    ax.axis('off')
    for k in range(4):
        rect = patches.Rectangle((0, offset[k]), 1, offset[k+1]-offset[k], linewidth=0, edgecolor='none', facecolor=cluster_color[k])
        ax.add_patch(rect)
        # ax.text(np.mean(offset[i:(i+2)]), -0.2, label, rotation=90, fontsize=10, horizontalalignment='left', verticalalignment='top')
    
fig.tight_layout()
fig.savefig('mCG_distribution/HemaB_10kb_hist_cross.pdf', transparent=True)


In [ ]:
import matplotlib.patches as patches

nc = 4
cluster_key = f'kmeans{nc}'
cluster_color = sns.color_palette('Set2', nc)
# leg_order = meta.groupby('L1')['mCGFrac'].median().sort_values().index

fig, axes = plt.subplots(2, 3, figsize=(4.2, 4), sharey='all', sharex='col', dpi=300, 
                         gridspec_kw={'width_ratios': [20,20,1]})

label = {}
for i,pmd_ct in enumerate(['c1', 'c15']):
    adata = anndata.read_h5ad(f'{outdir}{pmd_ct}_10kb_hist.h5ad')[selc]
    # label = adata.obs[cluster_key]
    idx = adata.obs[cluster_key].sort_values().index
    offset = list(adata.obs.loc[idx, cluster_key].value_counts().loc[leg].cumsum())
    offset = [0] + offset
    
    for j,data_ct in enumerate(['c1', 'c15']):
        ax = axes[i, j]
        sns.despine(ax=ax, left=True, bottom=True)
        adata = anndata.read_h5ad(f'{outdir}{data_ct}_10kb_hist.h5ad')
        tmp = adata[idx].X
        ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', 
                  interpolation='none', rasterized=True)
        if i==0:
            ax.set_title(L1_annot[data_ct])
        ax.set_xticks([-0.5, 49.5, 99.5])
        ax.set_xticklabels([0, 0.5, 1])
        ax.set_yticks([])
    
    ax = axes[i, -1]
    ax.axis('off')
    for k in range(4):
        rect = patches.Rectangle((0, offset[k]), 1, offset[k+1]-offset[k], linewidth=0, edgecolor='none', facecolor=cluster_color[k])
        ax.add_patch(rect)
        # ax.text(np.mean(offset[i:(i+2)]), -0.2, label, rotation=90, fontsize=10, horizontalalignment='left', verticalalignment='top')
    
fig.tight_layout()
fig.savefig('mCG_distribution/HemaT_10kb_hist_cross.pdf', transparent=True)


In [ ]:
outdir

In [ ]:
# import pybedtools
# import pyranges as pr

# for ct in L1_meta.index:
#     adata = anndata.read_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')
#     tmp = adata.obs.index[adata.obs[cluster_key]==2]
#     df = tmp.to_series().str.split('-', expand=True)
#     df.columns = ['Chromosome', 'Start', 'End']
#     df[['Start','End']] = df[['Start','End']].astype(int)
#     gr = pr.PyRanges(df)
#     merged_gr = gr.merge()
#     merged_gr.to_bed(f'{outdir}{ct}_hist_{cluster_key}.bed')
#     print(ct)
    

In [ ]:
from sklearn.metrics import pairwise_distances

def centerdist(ct):
    adata = anndata.read_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')
    centroid = pd.DataFrame(adata.X, index=adata.obs.index).groupby(adata.obs[cluster_key]).mean()
    dist = pairwise_distances(adata.X, centroid, metric='cosine')
    dist = pd.DataFrame(1 - dist, index=adata.obs.index, columns=centroid.index)
    return dist

def onehot(ct):
    adata = anndata.read_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')
    dist = np.zeros((adata.shape[0], 4))
    dist[(np.arange(adata.shape[0]), adata.obs[cluster_key])] = 1
    dist = pd.DataFrame(dist, index=adata.obs.index, columns=[f'{ct}-{i}' for i in range(4)])
    return dist

cpu = 20
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for i,ct in enumerate(L1_meta.index):
        future = executor.submit(
            onehot,
            ct=ct,
        )
        futures[future] = ct

    data = []
    for future in as_completed(futures):
        ct = futures[future]
        data.append(future.result())
        print(f'{ct} finished')

data = pd.concat(data, join='inner', axis=1)


In [ ]:
data.to_hdf(f'{outdir}merged_kmeans4_onehot.hdf', key='data')

In [ ]:
merged_label = pd.read_hdf(f'PMD/10kb/merged_10kb_kmeans4raw.hdf', key='data')

In [ ]:
def clustering(ct):
    nc = 4
    cluster_key = f'kmeans{nc}init'
    adata = anndata.read_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')[merged_label.index]
    centers = [adata[merged_label==i].X.mean(axis=0) for i in range(4)]
    adata.obs[cluster_key] = KMeans(n_clusters=nc, random_state=0, init=centers).fit(adata.X).labels_
    adata.write_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')
    return

cpu = 20
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for i,ct in enumerate(L1_meta.index):
        future = executor.submit(
            clustering,
            ct=ct,
        )
        futures[future] = ct

    for future in as_completed(futures):
        ct = futures[future]
        tmp = future.result()
        print(f'{ct} finished')


In [ ]:
import matplotlib.patches as patches

nc = 4
cluster_key = f'kmeans{nc}init'
cluster_color = sns.color_palette('Set2', nc)
leg_order = meta.groupby('L1')['mCGFrac'].median().sort_values().index
fig, axes = plt.subplots(6, 12, figsize=(13,12), sharey='all', sharex='col', dpi=300, 
                         gridspec_kw={'width_ratios': np.tile([20,1], 6)})
for i,ct in enumerate(leg_order):
    ax = axes.flatten()[i*2]
    sns.despine(ax=ax, left=True, bottom=True)
    adata = anndata.read_h5ad(f'{outdir}{ct}_10kb_hist.h5ad')
    
    leg = np.sort(adata.obs[cluster_key].unique())
    # selc = np.random.choice(np.arange(adata.shape[0]), 1000, False)
    # idx = adata.obs[cluster_key].iloc[selc].sort_values().index
    idx = adata.obs[cluster_key].sort_values().index
    tmp = adata[idx].X
    ax.imshow(tmp, cmap='vlag', vmax=np.percentile(tmp, 99), aspect='auto', 
              interpolation='none', rasterized=True)
    ax.set_title(L1_annot[ct])
    offset = list(adata.obs.loc[idx, cluster_key].value_counts().loc[leg].cumsum())
    offset = [0] + offset
    ax.set_xticks([-0.5, 49.5, 99.5])
    ax.set_xticklabels([0, 0.5, 1])
    ax.set_yticks([])
    
    ax = axes.flatten()[i*2+1]
    ax.axis('off')
    for k in range(4):
        rect = patches.Rectangle((0, offset[k]), 1, offset[k+1]-offset[k], linewidth=0, edgecolor='none', facecolor=cluster_color[k])
        ax.add_patch(rect)
        # ax.text(np.mean(offset[i:(i+2)]), -0.2, label, rotation=90, fontsize=10, horizontalalignment='left', verticalalignment='top')
    
fig.tight_layout()
fig.savefig('mCG_distribution/L1_10kb_hist_kmeans4init.pdf', transparent=True)
